# LeetCode 1004 — Max Consecutive Ones III bằng Python

Link đề: https://leetcode.com/problems/max-consecutive-ones-iii/ · Độ khó: **Medium** · Chủ đề: Array (mảng), Binary Search (tìm kiếm nhị phân), Sliding Window (cửa sổ trượt), Prefix Sum (tổng tiền tố)

Skeleton LeetCode:

```python
class Solution:
    def longestOnes(self, nums: List[int], k: int) -> int:
        ...
```

Ghi chú: cho mảng nhị phân `nums` và số nguyên `k`; được **đổi tối đa `k` số `0` thành `1`**. Trả về độ dài lớn nhất của đoạn **liên tiếp toàn số `1`** có thể thu được. Phát biểu lại không cần "đổi" gì cả: tìm **đoạn con dài nhất chứa tối đa `k` số `0`** (các số `0` trong đoạn đó sẽ được đổi thành `1`). Đó là bài mẫu của kỹ thuật **cửa sổ trượt**: mở rộng bờ phải, thu hẹp bờ trái khi số `0` vượt quá `k`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: "đổi tối đa `k` số `0`" tương đương "tìm đoạn con dài nhất có không quá `k` số `0`".
2. Nắm các công cụ Python cần thiết: duyệt `enumerate`, hai con trỏ `left`/`right`, `sum` và lát cắt, danh sách vị trí các số `0`, và `max` với biến đếm.
3. Hiểu nguyên lý của **cửa sổ trượt**: tính đơn điệu "bỏ bớt phần tử ở đầu chỉ làm số `0` ít đi", nên mỗi bờ trái chỉ tiến, tổng `O(n)`.
4. Đi qua 4 cách: duyệt mọi đoạn con (brute force), cửa sổ trượt co giãn (cách nộp chính), cửa sổ trượt **không co** (chỉ trượt) và cách dùng **danh sách vị trí các số `0`**.
5. Tránh các lỗi kinh điển: kiểm tra điều kiện thu hẹp **trước** khi thêm phần tử mới, quên trừ số `0` khi bờ trái đi qua, quên `+ 1` trong độ dài, xử lý sai `k = 0`, và đặt lại cửa sổ về rỗng như bài "Max Consecutive Ones" gốc.

## 2. Đề bài & ví dụ

Cho mảng nhị phân `nums` và số nguyên `k`, trả về **số lượng lớn nhất các số `1` liên tiếp** trong mảng nếu bạn được **đổi tối đa `k` số `0`** thành `1`.

**Ví dụ 1:** `nums = [1,1,1,0,0,0,1,1,1,1,0]`, `k = 2` -> `6`. Đổi hai số `0` (một ở vị trí 4, một ở vị trí 10) để được đoạn `[1,1,1,1,1,1]` từ vị trí 5 đến 10.

**Ví dụ 2:** `nums = [0,0,1,1,0,0,1,1,1,0,1,1,0,0,0,1,1,1,1]`, `k = 3` -> `10`.

**Ràng buộc:**

- `1 <= nums.length <= 10^5`
- `nums[i]` là `0` hoặc `1`.
- `0 <= k <= nums.length`

In [1]:
from typing import List
import random, time

examples = [
    ([1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0], 2, 6),
    ([0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1], 3, 10),
]
for nums, k, expected in examples:
    print(f"nums={nums} k={k} -> kỳ vọng {expected}")

nums=[1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0] k=2 -> kỳ vọng 6
nums=[0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1] k=3 -> kỳ vọng 10


## 3. Kiến thức nền

### 3.1. Đổi bài toán: "đoạn con dài nhất có tối đa `k` số `0`"

Một đoạn con liên tiếp `nums[l..r]` trở thành toàn `1` sau khi đổi các số `0` của nó. Ta tốn đúng bằng **số số `0`** trong đoạn. Vậy đoạn đó hợp lệ ⇔ `số 0 trong đoạn <= k`; và ta muốn **độ dài lớn nhất**. Không cần quan tâm các số ngoài đoạn.

In [2]:
nums = [1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0]
l, r = 5, 10
seg = nums[l:r + 1]
print("đoạn", seg, "có", seg.count(0), "số 0 -> cần đổi", seg.count(0), "số 0 (k = 2 đủ), độ dài", len(seg))

đoạn [0, 1, 1, 1, 1, 0] có 2 số 0 -> cần đổi 2 số 0 (k = 2 đủ), độ dài 6


### 3.2. Tính đơn điệu: vì sao cửa sổ trượt chạy được

Nếu đoạn `nums[l..r]` có `<= k` số `0` thì **mọi đoạn con của nó** cũng có `<= k` số `0`. Ngược lại nếu đoạn có `> k` số `0` thì mọi đoạn **chứa nó** cũng vậy. Nhờ vậy, khi bờ phải tăng và số `0` vượt `k`, ta chỉ cần **tăng bờ trái** cho tới khi hợp lệ lại, và **không bao giờ cần lùi** bờ trái.

In [3]:
nums = [1, 0, 1, 0, 1]
k = 1
def zeros(l, r): return nums[l:r + 1].count(0)
print("đoạn [0,2] có", zeros(0, 2), "số 0 (hợp lệ k=1); các đoạn con của nó:", [zeros(a, b) for a in range(0, 3) for b in range(a, 3)])
print("đoạn [0,4] có", zeros(0, 4), "số 0 (quá k=1); đoạn chứa nó cũng quá")

đoạn [0,2] có 1 số 0 (hợp lệ k=1); các đoạn con của nó: [0, 1, 1, 1, 1, 0]
đoạn [0,4] có 2 số 0 (quá k=1); đoạn chứa nó cũng quá


### 3.3. Cửa sổ trượt (sliding window)

Giữ hai con trỏ `left` và `right`; cửa sổ là `nums[left..right]`. Bước chuẩn: đưa `right` lên một ô, cập nhật thông tin (số `0`), **nếu vi phạm** thì đưa `left` lên cho tới khi hết vi phạm, rồi cập nhật đáp án bằng `right - left + 1`.

## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 1 với `k = 2`: `nums = [1,1,1,0,0,0,1,1,1,1,0]` (chỉ số `0..10`).

- Mở rộng `right` qua `0..4`: có `2` số `0` (vị trí 3, 4): cửa sổ `[0..4]` độ dài `5`, hợp lệ.
- `right = 5` (một số `0` nữa: tổng `3 > 2`): thu hẹp `left` từ `0` đến `4` (vì `nums[3] = 0` rời khỏi cửa sổ khi `left` vượt qua `3`): cửa sổ `[4..5]` có hai số `0`.
- `right = 6..9` (toàn `1`): cửa sổ `[4..9]` độ dài `6`, vẫn `2` số `0`.
- `right = 10` (số `0`): tổng `3 > 2`; thu hẹp `left` từ `4` lên `6` (qua số `0` ở 4, 5): cửa sổ `[6..10]` độ dài `5`.

Đáp án là độ dài lớn nhất đã gặp: `6`.

## 5. Cách 1 — Brute Force: thử mọi đoạn con

**Ý tưởng.** Với mỗi `left`, mở rộng `right` về phía phải và đếm số `0`; dừng khi số `0` vượt `k`; ghi nhận độ dài lớn nhất. Đây là định nghĩa trực tiếp ("đoạn con dài nhất có `<= k` số `0`") nên cũng là **oracle độc lập**: không dùng cửa sổ trượt hay tính đơn điệu theo hai con trỏ (chỉ dùng tính đơn điệu để dừng sớm khi mở rộng).

In [4]:
def longest_ones_brute(nums: List[int], k: int) -> int:
    n = len(nums)
    best = 0
    for left in range(n):
        zeros = 0
        for right in range(left, n):
            if nums[right] == 0:
                zeros += 1
            if zeros > k:
                break
            best = max(best, right - left + 1)
    return best

for nums, k, expected in examples:
    got = longest_ones_brute(nums, k)
    print(f"k={k}: brute = {got}", "OK" if got == expected else "SAI")
    assert got == expected

k=2: brute = 6 OK
k=3: brute = 10 OK


**Độ phức tạp và nút thắt.** `O(n^2)` tồi nhất (khi `k` lớn, mỗi `left` mở rộng gần hết mảng): với `n = 10^5` là `~ 5 * 10^9` phép toán. Đo thử:

In [5]:
rng = random.Random(1004)
for n in (500, 1000, 2000, 4000):
    nums = [rng.randint(0, 1) for _ in range(n)]
    t0 = time.perf_counter()
    longest_ones_brute(nums, n // 2)
    dt = time.perf_counter() - t0
    print(f"n={n:5}: {dt * 1000:8.1f} ms -> ngoại suy n=10^5: ~{dt * (10 ** 5 / n) ** 2:9.0f} s")

n=  500:     11.2 ms -> ngoại suy n=10^5: ~      450 s
n= 1000:     46.4 ms -> ngoại suy n=10^5: ~      464 s


n= 2000:    200.8 ms -> ngoại suy n=10^5: ~      502 s


n= 4000:    847.7 ms -> ngoại suy n=10^5: ~      530 s


Nút thắt: khi dời `left` sang `left + 1` ta **đếm lại từ đầu** các số `0` của đoạn, trong khi đoạn mới gần giống đoạn cũ. Hãy giữ số `0` và **chỉ cập nhật phần chênh lệch**.

## 6. Ý tưởng thuật toán / bước đột phá

**Giữ một cửa sổ `[left, right]` luôn có `<= k` số `0`; mỗi con trỏ chỉ tiến.**

- Với mỗi `right`: nếu `nums[right] == 0` thì tăng đếm `zeros`.
- Trong khi `zeros > k`: nếu `nums[left] == 0` thì giảm `zeros`; tăng `left`.
- Cập nhật đáp án `max(best, right - left + 1)`.

Mỗi chỉ số vào cửa sổ đúng một lần (khi `right` đi qua) và ra đúng một lần (khi `left` đi qua): `O(n)`.

| Cách | Ý tưởng |
|---|---|
| 2 | Cửa sổ trượt co giãn: `while zeros > k` thu hẹp (nộp) |
| 3 | Cửa sổ chỉ trượt (không co): kích thước cửa sổ không bao giờ giảm, đáp án là kích thước cuối |
| 4 | Danh sách vị trí các số `0`: đoạn dài nhất nằm giữa hai số `0` cách nhau `k + 1` vị trí trong danh sách |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Cửa sổ trượt co giãn (cách nộp chính)

**Giải thích từng đoạn.**

- `zeros`: số `0` trong cửa sổ `nums[left..right]`.
- Với mỗi `right`: cộng thêm nếu `nums[right] == 0`.
- `while zeros > k`: bỏ `nums[left]` khỏi cửa sổ (nếu là `0` thì trừ `zeros`), tăng `left`.
- Sau vòng `while`, cửa sổ hợp lệ; cập nhật `best`.

In [6]:
def longest_ones_window(nums: List[int], k: int) -> int:
    left = zeros = best = 0
    for right, x in enumerate(nums):
        if x == 0:
            zeros += 1
        while zeros > k:
            if nums[left] == 0:
                zeros -= 1
            left += 1
        best = max(best, right - left + 1)
    return best

for nums, k, expected in examples:
    got = longest_ones_window(nums, k)
    print(f"k={k}: cửa sổ trượt = {got}", "OK" if got == expected else "SAI")
    assert got == expected

k=2: cửa sổ trượt = 6 OK
k=3: cửa sổ trượt = 10 OK


**Code nộp LeetCode (Cách 2):**

In [7]:
class Solution:
    def longestOnes(self, nums: List[int], k: int) -> int:
        left = zeros = best = 0
        for right, x in enumerate(nums):
            if x == 0:
                zeros += 1                      # số 0 mới vào cửa sổ
            while zeros > k:                    # quá k số 0: thu hẹp từ trái
                if nums[left] == 0:
                    zeros -= 1
                left += 1
            best = max(best, right - left + 1)
        return best

sol_window = Solution().longestOnes
for nums, k, expected in examples:
    assert sol_window(nums, k) == expected
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** thời gian `O(n)` (mỗi con trỏ đi tối đa `n` bước), bộ nhớ `O(1)`.

---

### 7.2. Cách 3 — Cửa sổ chỉ trượt (không co)

**Ý tưởng khác.** Ta chỉ cần **độ dài** của cửa sổ tốt nhất, không cần cửa sổ luôn hợp lệ. Cho `right` tăng đều; khi số `0` vượt `k`, ta chỉ đẩy `left` **một** bước (cửa sổ giữ nguyên độ dài thay vì thu nhỏ). Độ dài cửa sổ **không bao giờ giảm**: nó chỉ tăng khi không vi phạm. Cuối cùng độ dài cửa sổ chính là đáp án (`n - left`).

**Giải thích từng đoạn.**

- Nếu `nums[right] == 0` tăng `zeros`.
- Nếu `zeros > k`: nếu `nums[left] == 0` giảm `zeros`; tăng `left` đúng một lần (`if`, không phải `while`).
- Cửa sổ `[left, right]` có thể tạm thời **không hợp lệ** (vẫn nhiều hơn `k` số `0`), nhưng nó không lớn hơn cửa sổ hợp lệ tốt nhất đã thấy; độ dài của nó chỉ tăng khi ta thật sự tìm được cửa sổ hợp lệ dài hơn.

In [8]:
def longest_ones_nonshrinking(nums: List[int], k: int) -> int:
    left = zeros = 0
    for right, x in enumerate(nums):
        if x == 0:
            zeros += 1
        if zeros > k:                           # chỉ trượt một bước: kích thước cửa sổ giữ nguyên
            if nums[left] == 0:
                zeros -= 1
            left += 1
    return len(nums) - left

for nums, k, expected in examples:
    got = longest_ones_nonshrinking(nums, k)
    print(f"k={k}: cửa sổ không co = {got}", "OK" if got == expected else "SAI")
    assert got == expected

k=2: cửa sổ không co = 6 OK
k=3: cửa sổ không co = 10 OK


**Code nộp LeetCode (Cách 3):**

In [9]:
class Solution:
    def longestOnes(self, nums: List[int], k: int) -> int:
        left = zeros = 0
        for right, x in enumerate(nums):
            if x == 0:
                zeros += 1
            if zeros > k:                        # trượt một bước (không thu nhỏ cửa sổ)
                if nums[left] == 0:
                    zeros -= 1
                left += 1
        return len(nums) - left                  # kích thước cửa sổ không bao giờ giảm

sol_nonshrinking = Solution().longestOnes
for nums, k, expected in examples:
    assert sol_nonshrinking(nums, k) == expected
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** thời gian `O(n)`, bộ nhớ `O(1)`; ít nhánh hơn Cách 2.

---

### 7.3. Cách 4 — Danh sách vị trí các số `0`

**Ý tưởng khác.** Đoạn dài nhất chứa tối đa `k` số `0` luôn có thể **mở rộng** cho tới khi chạm số `0` thứ `k + 1` (hoặc biên mảng). Gọi `zs` là danh sách vị trí các số `0` **kèm hai mốc chặn** `-1` ở đầu và `n` ở cuối. Một đoạn chứa **đúng `k` số `0` liên tiếp** `zs[i+1], ..., zs[i+k]` có thể mở rộng sang hai phía tới ngay trước `zs[i]` và `zs[i+k+1]`, nên độ dài là `zs[i + k + 1] - zs[i] - 1`. Đáp án là giá trị lớn nhất trên mọi `i` (và `n` nếu số `0` không vượt `k`).

**Giải thích từng đoạn.**

- `zs = [-1] + [vị trí các số 0] + [n]`; số số `0` là `m = len(zs) - 2`.
- Nếu `k >= m`: cả mảng chứa `<= k` số `0`: trả `n`.
- Ngược lại `i` chạy từ `0` đến `m - k`: `best = max(best, zs[i + k + 1] - zs[i] - 1)`.
- Khi `k = 0`: đoạn dài nhất toàn `1` nằm giữa hai số `0` liên tiếp: `zs[i + 1] - zs[i] - 1`.

In [10]:
def longest_ones_zero_positions(nums: List[int], k: int) -> int:
    n = len(nums)
    zs = [-1] + [i for i, x in enumerate(nums) if x == 0] + [n]
    m = len(zs) - 2
    if k >= m:
        return n
    return max(zs[i + k + 1] - zs[i] - 1 for i in range(m - k + 1))

for nums, k, expected in examples:
    got = longest_ones_zero_positions(nums, k)
    print(f"k={k}: vị trí số 0 = {got}", "OK" if got == expected else "SAI")
    assert got == expected

k=2: vị trí số 0 = 6 OK
k=3: vị trí số 0 = 10 OK


**Code nộp LeetCode (Cách 4):**

In [11]:
class Solution:
    def longestOnes(self, nums: List[int], k: int) -> int:
        n = len(nums)
        zs = [-1] + [i for i, x in enumerate(nums) if x == 0] + [n]   # vị trí các số 0 kèm hai mốc chặn
        m = len(zs) - 2                                                # số lượng số 0
        if k >= m:
            return n                                                   # đổi hết các số 0 được
        # đoạn chứa đúng k số 0 liên tiếp zs[i+1..i+k], mở rộng tới ngay trước zs[i] và zs[i+k+1]
        return max(zs[i + k + 1] - zs[i] - 1 for i in range(m - k + 1))

sol_zeros = Solution().longestOnes
for nums, k, expected in examples:
    assert sol_zeros(nums, k) == expected
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** thời gian `O(n)`, bộ nhớ `O(m)` với `m` là số số `0` (tới `O(n)`).

## 8. TRACE từng bước

### 8.1. Cách 2 cho ví dụ 1 (`k = 2`)

In `right`, `left`, `zeros` và độ dài cửa sổ sau mỗi bước.

In [12]:
def trace_window(nums, k):
    left = zeros = best = 0
    for right, x in enumerate(nums):
        if x == 0:
            zeros += 1
        moved = []
        while zeros > k:
            if nums[left] == 0:
                zeros -= 1
            moved.append(left)
            left += 1
        best = max(best, right - left + 1)
        print(f"right={right:2} x={x}: left={left:2} (đã bỏ chỉ số {moved or '-'}), zeros={zeros}, cửa sổ {nums[left:right + 1]}, độ dài {right - left + 1}, best={best}")
    return best

assert trace_window([1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0], 2) == 6

right= 0 x=1: left= 0 (đã bỏ chỉ số -), zeros=0, cửa sổ [1], độ dài 1, best=1
right= 1 x=1: left= 0 (đã bỏ chỉ số -), zeros=0, cửa sổ [1, 1], độ dài 2, best=2
right= 2 x=1: left= 0 (đã bỏ chỉ số -), zeros=0, cửa sổ [1, 1, 1], độ dài 3, best=3
right= 3 x=0: left= 0 (đã bỏ chỉ số -), zeros=1, cửa sổ [1, 1, 1, 0], độ dài 4, best=4
right= 4 x=0: left= 0 (đã bỏ chỉ số -), zeros=2, cửa sổ [1, 1, 1, 0, 0], độ dài 5, best=5
right= 5 x=0: left= 4 (đã bỏ chỉ số [0, 1, 2, 3]), zeros=2, cửa sổ [0, 0], độ dài 2, best=5
right= 6 x=1: left= 4 (đã bỏ chỉ số -), zeros=2, cửa sổ [0, 0, 1], độ dài 3, best=5
right= 7 x=1: left= 4 (đã bỏ chỉ số -), zeros=2, cửa sổ [0, 0, 1, 1], độ dài 4, best=5
right= 8 x=1: left= 4 (đã bỏ chỉ số -), zeros=2, cửa sổ [0, 0, 1, 1, 1], độ dài 5, best=5
right= 9 x=1: left= 4 (đã bỏ chỉ số -), zeros=2, cửa sổ [0, 0, 1, 1, 1, 1], độ dài 6, best=6
right=10 x=0: left= 5 (đã bỏ chỉ số [4]), zeros=2, cửa sổ [0, 1, 1, 1, 1, 0], độ dài 6, best=6


### 8.2. Cách 4 cho ví dụ 1: danh sách vị trí các số `0`

`zs = [-1, 3, 4, 5, 10, 11]` (`n = 11`). Với `k = 2` và `i = 0..2`: các độ dài `zs[i+3] - zs[i] - 1`.

In [13]:
nums, k = [1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0], 2
n = len(nums)
zs = [-1] + [i for i, x in enumerate(nums) if x == 0] + [n]
print("zs =", zs, "| số số 0 m =", len(zs) - 2)
for i in range(len(zs) - 2 - k + 1):
    print(f"i={i}: đoạn nằm giữa zs[{i}]={zs[i]} và zs[{i + k + 1}]={zs[i + k + 1]} -> độ dài {zs[i + k + 1] - zs[i] - 1}")
assert max(zs[i + k + 1] - zs[i] - 1 for i in range(len(zs) - 2 - k + 1)) == 6

zs = [-1, 3, 4, 5, 10, 11] | số số 0 m = 4
i=0: đoạn nằm giữa zs[0]=-1 và zs[3]=5 -> độ dài 5
i=1: đoạn nằm giữa zs[1]=3 và zs[4]=10 -> độ dài 6
i=2: đoạn nằm giữa zs[2]=4 và zs[5]=11 -> độ dài 6


### 8.3. Cách 3 (cửa sổ không co): kích thước cửa sổ chỉ tăng

In độ dài cửa sổ `right - left + 1` sau mỗi bước: dãy này **không giảm**, và giá trị cuối là đáp án.

In [14]:
def trace_nonshrinking(nums, k):
    left = zeros = 0
    sizes = []
    for right, x in enumerate(nums):
        if x == 0:
            zeros += 1
        if zeros > k:
            if nums[left] == 0:
                zeros -= 1
            left += 1
        sizes.append(right - left + 1)
    return sizes

sizes = trace_nonshrinking([1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0], 2)
print("kích thước cửa sổ sau mỗi bước:", sizes)
assert sizes == sorted(sizes) and sizes[-1] == 6

kích thước cửa sổ sau mỗi bước: [1, 2, 3, 4, 5, 5, 5, 5, 5, 6, 6]


## 9. Code nộp LeetCode

Ta chọn **Cách 2** (cửa sổ trượt co giãn): chuẩn mực, dễ chứng minh đúng.

```python
class Solution:
    def longestOnes(self, nums: List[int], k: int) -> int:
        left = zeros = best = 0
        for right, x in enumerate(nums):
            if x == 0:
                zeros += 1                      # số 0 mới vào cửa sổ
            while zeros > k:                    # quá k số 0: thu hẹp từ trái
                if nums[left] == 0:
                    zeros -= 1
                left += 1
            best = max(best, right - left + 1)
        return best
```

In [15]:
class Solution:
    def longestOnes(self, nums: List[int], k: int) -> int:
        left = zeros = best = 0
        for right, x in enumerate(nums):
            if x == 0:
                zeros += 1                      # số 0 mới vào cửa sổ
            while zeros > k:                    # quá k số 0: thu hẹp từ trái
                if nums[left] == 0:
                    zeros -= 1
                left += 1
            best = max(best, right - left + 1)
        return best

sol = Solution()
for nums, k, expected in examples:
    got = sol.longestOnes(nums, k)
    print(f"k={k}: {got}", "OK" if got == expected else "SAI")
    assert got == expected

k=2: 6 OK
k=3: 10 OK


## 10. Vì sao đúng

**Phát biểu tương đương.** Đáp án là độ dài lớn nhất của đoạn `nums[l..r]` có `<= k` số `0` (đổi các số `0` trong đó thành `1`; các số `0` ngoài đoạn không liên quan).

**Cách 2.** Bất biến: sau khi xử lý `right`, `left` là chỉ số **nhỏ nhất** sao cho `nums[left..right]` có `<= k` số `0`, và `zeros` đúng bằng số `0` trong cửa sổ.

- *Duy trì `zeros`:* thêm `nums[right]` và bỏ `nums[left]` đúng lúc nên `zeros` luôn khớp.
- *`left` nhỏ nhất:* với `right` cố định, tập các `left` hợp lệ là một **đoạn đuôi** (đơn điệu: dời `left` sang phải chỉ làm số `0` giảm). Hơn nữa `left_tối_ưu(right)` **không giảm** khi `right` tăng (nếu `[l, right + 1]` hợp lệ thì `[l, right]` cũng hợp lệ). Vòng `while` dừng ngay khi cửa sổ hợp lệ, nên `left` đúng bằng chỉ số nhỏ nhất đó; vì nó chỉ cần **tăng** nên không bao giờ phải lùi.
- *Đáp án:* với mỗi `right`, `right - left + 1` là độ dài đoạn hợp lệ dài nhất **kết thúc tại `right`**; lấy `max` theo `right` cho đoạn dài nhất toàn mảng.

**Cách 3.** Gọi `W_r` là kích thước cửa sổ sau bước `right`. Ta chứng minh `W_r = max(W_{r-1}, độ dài hợp lệ tốt nhất kết thúc tại r)`, tức kích thước cuối bằng đáp án. Khi bước `right` không gây vi phạm thì cửa sổ lớn thêm 1 chỉ khi cửa sổ cũ vẫn hợp lệ; khi gây vi phạm, ta đẩy `left` một bước nên kích thước giữ nguyên `W_{r-1}`. Cửa sổ không co chỉ tăng khi nó là hợp lệ, do đó kích thước cuối đúng bằng đoạn hợp lệ dài nhất. (Cửa sổ tạm thời không hợp lệ không làm tăng kích thước.)

**Cách 4.** Với đoạn tối ưu chứa `j <= k` số `0`: nếu `j < k` và còn số `0` kế tiếp ở ngoài thì ta mở rộng được thêm, nên tối ưu chứa đúng `min(k, m)` số `0`. Với đúng `k` số `0` là `zs[i+1..i+k]`, đoạn dài nhất là mọi vị trí nằm **giữa** `zs[i]` và `zs[i + k + 1]` (không chứa chúng): độ dài `zs[i+k+1] - zs[i] - 1`. Nếu `k >= m` thì cả mảng.

**Độ phức tạp.** Cách 2 và 3: mỗi con trỏ đi tối đa `n` bước nên `O(n)`.

**Kiểm chứng bằng máy.** Oracle (Cách 1) duyệt mọi đoạn con theo định nghĩa. Ta đối chiếu bốn cách trên mọi mảng nhị phân độ dài `<= 12` với mọi `k` có nghĩa (`0..n`), rồi trên nhiều mảng ngẫu nhiên có seed (nhiều mật độ số `0`).

In [16]:
from itertools import product
total = 0
for n in range(1, 13):
    for tup in product((0, 1), repeat=n):
        nums = list(tup)
        for k in range(0, n + 1):
            ref = longest_ones_brute(nums, k)
            assert longest_ones_window(nums, k) == ref, (nums, k)
            assert longest_ones_nonshrinking(nums, k) == ref, (nums, k)
            assert longest_ones_zero_positions(nums, k) == ref, (nums, k)
            total += 1
print("đã đối chiếu", total, "cặp (mảng nhị phân độ dài 1..12, k = 0..n): cả ba cách nhanh khớp oracle")

đã đối chiếu 98304 cặp (mảng nhị phân độ dài 1..12, k = 0..n): cả ba cách nhanh khớp oracle


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mọi đoạn con | `O(n^2)` | `O(1)` |
| 2. Cửa sổ co giãn | `O(n)` | `O(1)` |
| 3. Cửa sổ chỉ trượt | `O(n)` | `O(1)` |
| 4. Vị trí các số `0` | `O(n)` | `O(m)` |

Đo ở cỡ tối đa (`n = 10^5`) với ba dạng: ngẫu nhiên, toàn số `0` (cần thu hẹp nhiều), toàn số `1`:

In [17]:
rng = random.Random(11)
n = 10 ** 5
cases = {"ngẫu nhiên, k = 1000": ([rng.randint(0, 1) for _ in range(n)], 1000),
         "toàn số 0, k = 500": ([0] * n, 500),
         "toàn số 1, k = 0": ([1] * n, 0),
         "xen kẽ, k = 0": ([i % 2 for i in range(n)], 0)}
for label, (nums, k) in cases.items():
    ref = sol_window(nums, k)
    for name, fn in (("Cách 2", sol_window), ("Cách 3", sol_nonshrinking), ("Cách 4", sol_zeros), ("nộp", sol.longestOnes)):
        t0 = time.perf_counter()
        got = fn(nums, k)
        dt = (time.perf_counter() - t0) * 1000
        assert got == ref, name
        print(f"{label:24} {name:7}: {dt:7.2f} ms, đáp án {got}")

ngẫu nhiên, k = 1000     Cách 2 :   29.49 ms, đáp án 2149
ngẫu nhiên, k = 1000     Cách 3 :   17.45 ms, đáp án 2149
ngẫu nhiên, k = 1000     Cách 4 :   18.62 ms, đáp án 2149
ngẫu nhiên, k = 1000     nộp    :   30.15 ms, đáp án 2149


toàn số 0, k = 500       Cách 2 :   32.31 ms, đáp án 500
toàn số 0, k = 500       Cách 3 :   19.15 ms, đáp án 500
toàn số 0, k = 500       Cách 4 :   28.52 ms, đáp án 500
toàn số 0, k = 500       nộp    :   29.59 ms, đáp án 500


toàn số 1, k = 0         Cách 2 :   17.50 ms, đáp án 100000
toàn số 1, k = 0         Cách 3 :    6.97 ms, đáp án 100000
toàn số 1, k = 0         Cách 4 :    5.55 ms, đáp án 100000
toàn số 1, k = 0         nộp    :   17.28 ms, đáp án 100000
xen kẽ, k = 0            Cách 2 :   13.67 ms, đáp án 1
xen kẽ, k = 0            Cách 3 :   11.33 ms, đáp án 1
xen kẽ, k = 0            Cách 4 :   12.33 ms, đáp án 1


xen kẽ, k = 0            nộp    :   16.51 ms, đáp án 1


## 12. Bộ test

Gồm: ví dụ của đề, edge case có kỳ vọng viết tay (kèm lý do), phản ví dụ cho các cách sai (tìm bằng tìm kiếm so với oracle), đối chiếu mọi cách, test ngẫu nhiên có seed (cả cỡ tối đa) và các tính chất.

In [18]:
def all_solutions():
    return {"window": longest_ones_window, "nonshrinking": longest_ones_nonshrinking, "zeros": longest_ones_zero_positions,
            "sol_window": sol_window, "sol_nonshrinking": sol_nonshrinking, "sol_zeros": sol_zeros, "final": sol.longestOnes}

def check_all(nums, k, expected=None, brute=True):
    results = {name: fn(list(nums), k) for name, fn in all_solutions().items()}
    assert len(set(results.values())) == 1, (nums[:20], k, results)
    ref = results["window"]
    if brute and len(nums) <= 3000:
        assert longest_ones_brute(nums, k) == ref, (nums[:20], k)
    if expected is not None:
        assert ref == expected, (nums, k, ref, expected)
    return ref

# 12.1 ví dụ đề
for nums, k, expected in examples:
    check_all(nums, k, expected)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    ([0], 0, 0),                        # một số 0, không được đổi
    ([0], 1, 1),                        # đổi được số 0 duy nhất
    ([1], 0, 1),                        # một số 1
    ([1], 1, 1),                        # dư lượt đổi
    ([0, 0, 0, 0, 0], 2, 2),            # toàn số 0: chỉ đổi được 2
    ([0, 0, 0, 0, 0], 5, 5),            # đổi hết
    ([1, 1, 1, 1], 0, 4),               # toàn số 1
    ([1, 0, 1], 0, 1),                  # k = 0: đoạn toàn 1 dài nhất
    ([1, 0, 1], 1, 3),                  # đổi số 0 ở giữa
    ([1, 0, 1, 0, 1], 1, 3),            # đổi một số 0, hai bên mỗi bên một số 1
    ([1, 0, 1, 0, 1], 2, 5),            # đổi cả hai số 0
    ([0, 1, 1, 1, 0], 0, 3),            # k = 0, khối giữa
    ([0, 1, 1, 1, 0], 1, 4),            # k = 1: bắt một đầu
    ([1, 1, 0, 0, 1, 1, 1], 1, 4),      # k = 1: 0 + 111 hoặc 11 + 0
    ([1, 1, 0, 0, 1, 1, 1], 2, 7),      # k = 2: cả mảng
    ([1, 0, 0, 1, 1, 0, 1], 1, 4),      # tính tay ở bài tập 2: [1,1,0,1]
    ([0, 1, 0, 1, 0, 1, 0], 2, 5),      # xen kẽ
    ([1] * 10 + [0] * 3 + [1] * 10, 3, 23),   # lấp đầy ba số 0 liên tiếp
    ([1] * 10 + [0] * 3 + [1] * 10, 2, 12),   # chỉ đổi được hai số 0: một bên 10 + 2 số 0 đã đổi
]
for nums, k, expected in edge:
    check_all(nums, k, expected)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 19 ca


In [19]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm, so với oracle brute)
def wrong_check_before_add(nums, k):             # thu hẹp TRƯỚC khi thêm phần tử mới: cửa sổ có thể chứa k + 1 số 0
    left = zeros = best = 0
    for right, x in enumerate(nums):
        while zeros > k:
            if nums[left] == 0:
                zeros -= 1
            left += 1
        if x == 0:
            zeros += 1
        best = max(best, right - left + 1)
    return best

def wrong_no_decrement(nums, k):                  # quên trừ zeros khi left đi qua số 0
    left = zeros = best = 0
    for right, x in enumerate(nums):
        if x == 0:
            zeros += 1
        while zeros > k and left <= right:
            left += 1
        best = max(best, right - left + 1)
    return best

def wrong_off_by_one(nums, k):                    # độ dài = right - left (thiếu +1)
    left = zeros = best = 0
    for right, x in enumerate(nums):
        if x == 0:
            zeros += 1
        while zeros > k:
            if nums[left] == 0:
                zeros -= 1
            left += 1
        best = max(best, right - left)
    return best

def wrong_reset(nums, k):                         # như bài Max Consecutive Ones gốc: gặp 0 quá k thì đặt lại về 0
    zeros = best = cur = 0
    for x in nums:
        if x == 1:
            cur += 1
        else:
            zeros += 1
            if zeros <= k:
                cur += 1
            else:
                cur, zeros = 0, 0
        best = max(best, cur)
    return best

def wrong_k_zero(nums, k):                        # coi k = 0 như k = 1 (dùng k or 1)
    return longest_ones_window(nums, k or 1)

def wrong_greedy_flip_first(nums, k):             # đổi k số 0 đầu tiên rồi đếm đoạn 1 dài nhất
    a = list(nums)
    flipped = 0
    for i, x in enumerate(a):
        if x == 0 and flipped < k:
            a[i] = 1
            flipped += 1
    best = cur = 0
    for x in a:
        cur = cur + 1 if x == 1 else 0
        best = max(best, cur)
    return best

def find_counter(wrong, max_len=8):
    for n in range(1, max_len + 1):
        for tup in product((0, 1), repeat=n):
            nums = list(tup)
            for k in range(0, n + 1):
                if wrong(list(nums), k) != longest_ones_brute(nums, k):
                    return nums, k
    return None

counters = {}
for name, fn in (("kiểm tra thu hẹp trước khi thêm phần tử", wrong_check_before_add), ("quên trừ zeros khi left qua số 0", wrong_no_decrement),
                 ("thiếu +1 trong độ dài", wrong_off_by_one), ("đặt lại cửa sổ về 0", wrong_reset),
                 ("coi k = 0 như k = 1", wrong_k_zero), ("đổi k số 0 đầu tiên", wrong_greedy_flip_first)):
    case = find_counter(fn)
    assert case is not None, name
    counters[name] = case
    nums, k = case
    print(f"{name:44} nums={nums} k={k}: sai -> {fn(list(nums), k)}, đúng -> {longest_ones_brute(nums, k)}")
    check_all(nums, k)
print("12.3 phản ví dụ: OK")

kiểm tra thu hẹp trước khi thêm phần tử      nums=[0] k=0: sai -> 1, đúng -> 0
quên trừ zeros khi left qua số 0             nums=[0, 1] k=0: sai -> 0, đúng -> 1
thiếu +1 trong độ dài                        nums=[0] k=1: sai -> 0, đúng -> 1
đặt lại cửa sổ về 0                          nums=[0, 0, 1] k=1: sai -> 1, đúng -> 2
coi k = 0 như k = 1                          nums=[0] k=0: sai -> 1, đúng -> 0
đổi k số 0 đầu tiên                          nums=[0, 0, 1] k=1: sai -> 1, đúng -> 2
12.3 phản ví dụ: OK


In [20]:
# 12.4 ngẫu nhiên: nhiều mật độ số 0, k bất kỳ, n tới 3000 so với oracle; n lớn so các cách nhanh với nhau
rng = random.Random(4004)
for _ in range(600):
    n = rng.randint(1, 60)
    p = rng.choice([0.1, 0.3, 0.5, 0.8])
    nums = [0 if rng.random() < p else 1 for _ in range(n)]
    k = rng.randint(0, n)
    check_all(nums, k)
for _ in range(10):
    n = rng.randint(1000, 3000)
    nums = [rng.randint(0, 1) for _ in range(n)]
    check_all(nums, rng.randint(0, n))
print("12.4 ngẫu nhiên: OK")

rng = random.Random(5004)
for _ in range(20):
    n = rng.randint(50000, 100000)
    p = rng.choice([0.05, 0.5, 0.95])
    nums = [0 if rng.random() < p else 1 for _ in range(n)]
    check_all(nums, rng.choice([0, 1, 10, n // 100, n // 2, n]), brute=False)
print("12.5 cỡ lớn (n tới 10^5): OK")

# 12.6 tính chất: đơn điệu theo k; k = 0 là đoạn 1 dài nhất; k >= số 0 cho n; đảo mảng không đổi; thêm k tăng tối đa ... ; k = số 0 ở một phía
rng = random.Random(6004)
for _ in range(300):
    n = rng.randint(1, 80)
    nums = [rng.randint(0, 1) for _ in range(n)]
    vals = [sol.longestOnes(nums, k) for k in range(0, n + 1)]
    assert vals == sorted(vals)                                    # không giảm theo k
    run = best = 0
    for x in nums:
        run = run + 1 if x == 1 else 0
        best = max(best, run)
    assert vals[0] == best                                         # k = 0: chuỗi 1 dài nhất
    assert vals[nums.count(0)] == n and vals[n] == n               # đủ k để đổi hết
    assert sol.longestOnes(nums[::-1], 3) == sol.longestOnes(nums, 3)   # đối xứng khi đảo
    assert all(vals[k] >= min(n, k) for k in range(n + 1))                # đổi k số liền nhau: ít nhất min(n, k)
print("12.6 tính chất: OK")

12.4 ngẫu nhiên: OK


12.5 cỡ lớn (n tới 10^5): OK
12.6 tính chất: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Kiểm tra vi phạm **trước** khi thêm phần tử mới

Nếu thu hẹp cửa sổ rồi mới thêm `nums[right]`, thì sau khi thêm cửa sổ có thể chứa `k + 1` số `0` nhưng ta vẫn ghi nhận độ dài của nó. (Lưu ý: thu hẹp bằng `if` rồi trả `n - left` như Cách 3 thì **đúng**; vấn đề ở đây là thứ tự thêm/thu hẹp.)

In [21]:
nums, k = counters["kiểm tra thu hẹp trước khi thêm phần tử"]
print("nums =", nums, "k =", k, "| sai ->", wrong_check_before_add(nums, k), "| đúng ->", Solution().longestOnes(nums, k))
assert wrong_check_before_add(nums, k) != Solution().longestOnes(nums, k)

nums = [0] k = 0 | sai -> 1 | đúng -> 0


Vì sao sai: lúc ghi nhận độ dài, cửa sổ vừa thêm phần tử mới (một số `0`) và có thể đã vượt `k`. **Đúng:** thêm phần tử mới **trước**, rồi mới `while zeros > k` thu hẹp, rồi ghi nhận.

### Lỗi 2 — Quên trừ số `0` khi `left` đi qua

Nếu chỉ tăng `left` mà không trừ `zeros` khi gặp `nums[left] == 0` thì `zeros` không bao giờ giảm.

In [22]:
nums, k = counters["quên trừ zeros khi left qua số 0"]
print("nums =", nums, "k =", k, "| sai ->", wrong_no_decrement(nums, k), "| đúng ->", Solution().longestOnes(nums, k))
assert wrong_no_decrement(nums, k) != Solution().longestOnes(nums, k)

nums = [0, 1] k = 0 | sai -> 0 | đúng -> 1


Vì sao sai: `zeros` phản ánh số `0` **trong cửa sổ**; khi `left` rời khỏi một số `0` phải trừ. **Đúng:** `if nums[left] == 0: zeros -= 1` trước `left += 1`.

### Lỗi 3 — Thiếu `+ 1` trong độ dài

Cửa sổ `[left, right]` có `right - left + 1` phần tử.

In [23]:
nums, k = counters["thiếu +1 trong độ dài"]
print("nums =", nums, "k =", k, "| sai ->", wrong_off_by_one(nums, k), "| đúng ->", Solution().longestOnes(nums, k))
assert wrong_off_by_one(nums, k) != Solution().longestOnes(nums, k)

nums = [0] k = 1 | sai -> 0 | đúng -> 1


Vì sao sai: lỗi lệch một đơn vị kinh điển (đếm khoảng cách thay vì số phần tử). **Đúng:** `right - left + 1`.

### Lỗi 4 — Đặt lại cửa sổ về `0` (như bài Max Consecutive Ones gốc)

Ở bài gốc (485) gặp `0` thì đặt lại; ở đây ta được **đổi** `k` số `0`, và khi vượt quá `k` thì chỉ cần **bỏ bớt phần đầu**, không bỏ cả cửa sổ.

In [24]:
nums, k = counters["đặt lại cửa sổ về 0"]
print("nums =", nums, "k =", k, "| sai ->", wrong_reset(nums, k), "| đúng ->", Solution().longestOnes(nums, k))
assert wrong_reset(nums, k) != Solution().longestOnes(nums, k)

nums = [0, 0, 1] k = 1 | sai -> 1 | đúng -> 2


Vì sao sai: bỏ cả cửa sổ làm mất các số `1` nằm **sau** số `0` bị bỏ. **Đúng:** thu hẹp từ trái cho tới khi `zeros <= k`.

### Lỗi 5 — Xử lý sai `k = 0`

Ví dụ coi `k = 0` như `k = 1` (`k or 1`) hay giả định `k > 0` trong vòng lặp.

In [25]:
nums, k = counters["coi k = 0 như k = 1"]
print("nums =", nums, "k =", k, "| sai ->", wrong_k_zero(nums, k), "| đúng ->", Solution().longestOnes(nums, k))
assert wrong_k_zero(nums, k) != Solution().longestOnes(nums, k)
nums, k = counters["đổi k số 0 đầu tiên"]
print("(đổi k số 0 đầu tiên) nums =", nums, "k =", k, "| sai ->", wrong_greedy_flip_first(nums, k), "| đúng ->", Solution().longestOnes(nums, k))
assert wrong_greedy_flip_first(nums, k) != Solution().longestOnes(nums, k)

nums = [0] k = 0 | sai -> 1 | đúng -> 0
(đổi k số 0 đầu tiên) nums = [0, 0, 1] k = 1 | sai -> 1 | đúng -> 2


Vì sao sai: `k = 0` là trường hợp hợp lệ (không đổi số nào): đáp án là chuỗi `1` dài nhất. Còn đổi `k` số `0` **đầu tiên** là tham lam sai vì vị trí đổi tối ưu phụ thuộc cửa sổ. **Đúng:** thuật toán cửa sổ trượt xử lý `k = 0` tự nhiên (`zeros > 0` là vi phạm).

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "đoạn con liên tiếp dài nhất sao cho một đại lượng (số `0`, số loại phần tử khác, tổng...) **không vượt** ngưỡng" ⇒ **cửa sổ trượt** (cần tính đơn điệu: bỏ bớt phần tử không làm đại lượng tăng).
- Khuôn mẫu: `for right in ...: thêm phần tử; while vi phạm: bỏ phần tử ở left; cập nhật đáp án`.
- Biến thể gọn: cửa sổ **không co** (`if` + trả `n - left`) khi chỉ cần **độ dài lớn nhất**.
- Bài toán "đổi tối đa `k`" thường là "chứa tối đa `k` phần tử xấu".
- Cùng họ: Max Consecutive Ones (485), Max Consecutive Ones II (487), Longest Repeating Character Replacement (424), Longest Subarray of 1's After Deleting One Element (1493), Fruit Into Baskets (904).
- Test nên có: `k = 0`, `k = n`, toàn `0`, toàn `1`, xen kẽ, `n = 1`.

## 15. Biến thể

1. **Xóa đúng một phần tử** rồi lấy đoạn `1` dài nhất (1493): bằng đáp án với `k = 1` trừ đi `1` (bài tập 5).
2. **Thay tối đa `k` ký tự để chuỗi toàn một ký tự** (424): cửa sổ hợp lệ khi `độ dài - tần suất lớn nhất <= k`.
3. **Đếm số đoạn con** có đúng/tối đa `k` số `0`: dùng `atMost(k) - atMost(k - 1)`.
4. Bài liên quan: 485, 487, 424, 1493, 904.

Biến thể 2 (Longest Repeating Character Replacement) kiểm bằng oracle duyệt mọi đoạn con:

In [26]:
def character_replacement(s: str, k: int) -> int:
    from collections import Counter
    cnt = Counter()
    left = best = max_freq = 0
    for right, ch in enumerate(s):
        cnt[ch] += 1
        max_freq = max(max_freq, cnt[ch])                 # tần suất lớn nhất đã gặp (không cần giảm)
        if (right - left + 1) - max_freq > k:
            cnt[s[left]] -= 1
            left += 1
        best = max(best, right - left + 1)
    return best

def character_replacement_brute(s: str, k: int) -> int:
    from collections import Counter
    best = 0
    for i in range(len(s)):
        cnt = Counter()
        for j in range(i, len(s)):
            cnt[s[j]] += 1
            if (j - i + 1) - max(cnt.values()) <= k:
                best = max(best, j - i + 1)
    return best

rng = random.Random(15)
for _ in range(400):
    s = "".join(rng.choice("ABC") for _ in range(rng.randint(1, 14)))
    k = rng.randint(0, len(s))
    assert character_replacement(s, k) == character_replacement_brute(s, k), (s, k)
print(character_replacement("AABABBA", 1), "(ví dụ của bài 424 cho 4)")
assert character_replacement("AABABBA", 1) == 4 and character_replacement("ABAB", 2) == 4

4 (ví dụ của bài 424 cho 4)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi đoạn con | Theo định nghĩa | `O(n^2)` | `O(1)` | Oracle |
| 2. Cửa sổ co giãn | `while zeros > k` thu hẹp | `O(n)` | `O(1)` | **Cách nộp chính** |
| 3. Cửa sổ chỉ trượt | `if` + trả `n - left` | `O(n)` | `O(1)` | Khi chỉ cần độ dài |
| 4. Vị trí các số `0` | Đoạn giữa hai số `0` cách `k + 1` vị trí | `O(n)` | `O(m)` | Cách nhìn khác, dễ chứng minh |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán rồi chạy để kiểm tra:

```python
print([1, 0, 1, 1, 0].count(0), sum([1, 0, 1, 1, 0]), len([1, 0, 1, 1, 0]) - sum([1, 0, 1, 1, 0]))
print(Solution().longestOnes([1, 0, 1, 1, 0], 0), Solution().longestOnes([1, 0, 1, 1, 0], 1), Solution().longestOnes([1, 0, 1, 1, 0], 2))
```

In [27]:
# Bài 1: ghi dự đoán của bạn
my_guess_1 = None      # ba số
my_guess_2 = None      # ba số

### Bài 2 — Tính tay (dễ-vừa)

`nums = [1, 0, 0, 1, 1, 0, 1]`, `k = 1`. Ghi đáp án (độ dài lớn nhất).

In [28]:
# Bài 2: điền kết quả tính tay
hand_answer = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành cửa sổ trượt.

In [29]:
def exercise_3(nums: List[int], k: int) -> int:
    left = zeros = best = 0
    for right, x in enumerate(nums):
        # TODO 1: nếu x == 0 thì tăng zeros
        # TODO 2: while zeros > k: nếu nums[left] == 0 thì giảm zeros; tăng left
        # TODO 3: cập nhật best với độ dài cửa sổ (right - left + 1)
        pass
    return best


# for nums, k, expected in examples:
#     assert exercise_3(nums, k) == expected

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 4, tự viết `exercise_4(nums, k)` bằng **danh sách vị trí các số `0`** (kèm mốc chặn `-1` và `n`).

In [30]:
def exercise_4(nums: List[int], k: int) -> int:
    # TODO
    pass


# for nums, k, expected in examples:
#     assert exercise_4(nums, k) == expected

### Bài 5 — Mini challenge: xóa đúng một phần tử (khó)

Viết `exercise_5(nums)` trả về độ dài đoạn con **toàn `1`** dài nhất sau khi **xóa đúng một phần tử** của `nums` (bài 1493; bạn **bắt buộc** xóa đúng một phần tử, kể cả khi toàn `1`; `len(nums) >= 1`).

```python
exercise_5([1, 1, 0, 1])                   -> 3
exercise_5([0, 1, 1, 1, 0, 1, 1, 0, 1])    -> 5
exercise_5([1, 1, 1])                      -> 2
exercise_5([0])                            -> 0
```
Gợi ý: đoạn dài nhất có tối đa một số `0`, rồi trừ `1` (phần tử bị xóa).

In [31]:
def exercise_5(nums: List[int]) -> int:
    # TODO
    pass


# assert exercise_5([1, 1, 0, 1]) == 3
# assert exercise_5([0, 1, 1, 1, 0, 1, 1, 0, 1]) == 5
# assert exercise_5([1, 1, 1]) == 2
# assert exercise_5([0]) == 0

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `count(0)` đếm số `0`; `sum` đếm số `1`.
- Mức 2: mảng `[1,0,1,1,0]` có 2 số `0`, 3 số `1`; với `k = 0`, đoạn `1` dài nhất là `[1, 1]`.
- Mức 3: dòng 1: `2 3 2`; dòng 2: `2 4 5`.

**Bài 2**
- Mức 1: đoạn dài nhất có tối đa 1 số `0`.
- Mức 2: các số `0` ở chỉ số 1, 2, 5; xét đoạn chứa đúng một số `0`.
- Mức 3: đoạn `[1, 1, 0, 1]` (chỉ số 3..6) có độ dài `4`; không có đoạn nào dài hơn: `4`.

**Bài 3**
- Mức 1: `if x == 0: zeros += 1`.
- Mức 2: `while zeros > k: if nums[left] == 0: zeros -= 1` rồi `left += 1`.
- Mức 3: `best = max(best, right - left + 1)`.

**Bài 4**
- Mức 1: `zs = [-1] + [i for i, x in enumerate(nums) if x == 0] + [n]`.
- Mức 2: `m = len(zs) - 2`; nếu `k >= m` trả `n`.
- Mức 3: `max(zs[i + k + 1] - zs[i] - 1 for i in range(m - k + 1))`.

**Bài 5**
- Mức 1: cửa sổ trượt với `k = 1`.
- Mức 2: trong đoạn có tối đa một số `0`, xóa số `0` (hoặc một số `1` nếu không có số `0`).
- Mức 3: trả về `longestOnes(nums, 1) - 1`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [32]:
print([1, 0, 1, 1, 0].count(0), sum([1, 0, 1, 1, 0]), len([1, 0, 1, 1, 0]) - sum([1, 0, 1, 1, 0]))
print(Solution().longestOnes([1, 0, 1, 1, 0], 0), Solution().longestOnes([1, 0, 1, 1, 0], 1), Solution().longestOnes([1, 0, 1, 1, 0], 2))

assert ([1, 0, 1, 1, 0].count(0), sum([1, 0, 1, 1, 0]), len([1, 0, 1, 1, 0]) - sum([1, 0, 1, 1, 0])) == (2, 3, 2)
assert (Solution().longestOnes([1, 0, 1, 1, 0], 0), Solution().longestOnes([1, 0, 1, 1, 0], 1), Solution().longestOnes([1, 0, 1, 1, 0], 2)) == (2, 4, 5)

2 3 2
2 4 5


### Đáp án Bài 2

`nums = [1,0,0,1,1,0,1]`, `k = 1`. Các đoạn chứa tối đa một số `0`: `[1,0]` (2), `[0,1,1]` (3), `[1,1,0,1]` (4, chỉ số 3..6), `[0,1,1,0]` có hai số `0` nên bị loại. Lớn nhất `4`.

In [33]:
hand_answer_solution = 4
assert Solution().longestOnes([1, 0, 0, 1, 1, 0, 1], 1) == hand_answer_solution
assert longest_ones_brute([1, 0, 0, 1, 1, 0, 1], 1) == hand_answer_solution
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [34]:
def exercise_3_solution(nums: List[int], k: int) -> int:
    left = zeros = best = 0
    for right, x in enumerate(nums):
        if x == 0:
            zeros += 1
        while zeros > k:
            if nums[left] == 0:
                zeros -= 1
            left += 1
        best = max(best, right - left + 1)
    return best

for nums, k, expected in examples:
    assert exercise_3_solution(nums, k) == expected
rng = random.Random(3)
for _ in range(300):
    n = rng.randint(1, 30)
    nums = [rng.randint(0, 1) for _ in range(n)]
    k = rng.randint(0, n)
    assert exercise_3_solution(nums, k) == longest_ones_brute(nums, k)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [35]:
def exercise_4_solution(nums: List[int], k: int) -> int:
    n = len(nums)
    zs = [-1] + [i for i, x in enumerate(nums) if x == 0] + [n]
    m = len(zs) - 2
    if k >= m:
        return n
    return max(zs[i + k + 1] - zs[i] - 1 for i in range(m - k + 1))

for nums, k, expected in examples:
    assert exercise_4_solution(nums, k) == expected
rng = random.Random(4)
for _ in range(300):
    n = rng.randint(1, 30)
    nums = [rng.randint(0, 1) for _ in range(n)]
    k = rng.randint(0, n)
    assert exercise_4_solution(nums, k) == longest_ones_brute(nums, k)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [36]:
def exercise_5_solution(nums: List[int]) -> int:
    return Solution().longestOnes(nums, 1) - 1

def delete_one_brute(nums):
    best = 0
    for d in range(len(nums)):
        rest = nums[:d] + nums[d + 1:]
        run = cur = 0
        for x in rest:
            cur = cur + 1 if x == 1 else 0
            run = max(run, cur)
        best = max(best, run)
    return best

assert exercise_5_solution([1, 1, 0, 1]) == 3
assert exercise_5_solution([0, 1, 1, 1, 0, 1, 1, 0, 1]) == 5
assert exercise_5_solution([1, 1, 1]) == 2
assert exercise_5_solution([0]) == 0
rng = random.Random(5)
for _ in range(400):
    nums = [rng.randint(0, 1) for _ in range(rng.randint(1, 14))]
    assert exercise_5_solution(nums) == delete_one_brute(nums), nums
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (cửa sổ trượt)

```python
class Solution:
    def longestOnes(self, nums: List[int], k: int) -> int:
        left = zeros = best = 0
        for right, x in enumerate(nums):
            if x == 0:
                zeros += 1                      # thêm phần tử phải
            while zeros > k:                    # vi phạm: thu hẹp từ trái
                if nums[left] == 0:
                    zeros -= 1
                left += 1
            best = max(best, right - left + 1)  # độ dài cửa sổ = right - left + 1
        return best
```

Công thức nhớ nhanh: đáp án là độ dài đoạn con dài nhất có `<= k` số `0`.

### Checklist trước khi submit

- [ ] Phát biểu lại thành "đoạn con dài nhất có tối đa `k` số `0`"?
- [ ] Dùng `while zeros > k` (không `if`) khi vừa thu hẹp vừa ghi nhận đáp án mỗi bước?
- [ ] Trừ `zeros` khi `left` đi qua một số `0`?
- [ ] Độ dài là `right - left + 1`?
- [ ] `k = 0` và `k = n` đều đúng (không có giả định `k > 0`)?
- [ ] Đã thử: `[0]`, `[1]`, toàn `0`, toàn `1`, xen kẽ, `k = 0`, `n = 10^5`?